### Libraries & Dependencies
In this section, we import the required libraries and tools for the project. These include libraries for web requests (Requests), HTML parsing (BeautifulSoup), working with Large Language Models (like Ollama/Llama), and building the user interface (Gradio).

In [15]:
import os
import json
from dotenv import load_dotenv
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI
import gradio as gr

### Checking Local LLM Status via Docker
In this cell, we run a system command to check the status of the Ollama Docker container. This ensures that the required models (such as llama3.2 and deepseek-r1) are properly installed, loaded, and ready to respond.

In [32]:
!docker exec -i main_ollama ollama list

NAME                ID              SIZE      MODIFIED    
deepseek-r1:1.5b    e0979632db5a    1.1 GB    7 days ago     
mistral:latest      6577803aa9a0    4.4 GB    9 days ago     
llama3.2:latest     a80c4f17acd5    2.0 GB    4 weeks ago    
gpt-oss:latest      17052f91a42e    13 GB     4 weeks ago    


### Environment Configuration
This section handles importing core packages and loading environment variables using load_dotenv to securely manage API keys or local configurations for connecting to the AI backend.

In [17]:
load_dotenv(override=True)
ollama_base_url = os.getenv("OLLAMA_BASE_URL")
ollama_model = os.getenv("OLLAMA_MODEL")
ollama = OpenAI(base_url=ollama_base_url, api_key='ollama')


### Web Scraping & Link Selection
This part of the code is responsible for receiving the company's main URL, identifying available links (such as "About Us", "Investors", and "Contact"), and then using a lightweight LLM to select the most relevant links for creating the brochure and extract their textual content.

### Generating Link Selection Prompts
At this stage, the core scraping function requests and parses the target website's HTML to pull out all raw internal and external links found on the homepage.

In [18]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [19]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    print(f"Generated user prompt for relevant links for {url}"+"\n")
    return user_prompt

### Calling LLM for Link Filtering
In this section, we communicate with the llama3.2 model. The generated prompt is sent to the LLM, which filters and selects the most optimal pages needed for the brochure.

In [20]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {ollama_model}")
    response = ollama.chat.completions.create(
        model=ollama_model,
        messages=[
            {"role": "user", "content": link_system_prompt + get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links"+"\n")
    return links
    

### Deep Content Fetching from Selected Links
Once the target links are selected by the AI, this function automatically iterates through each URL, extracts their main textual content, and merges them for the final processing pipeline.

In [21]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
        print(f"Finished fetching contents for {link['type']} at {link['url']}")
    print(f"Finished fetching contents for all relevant links {url} and sublinks")
    return result

In [22]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


### Text Truncation & Prompt Optimization
Due to context length limitations in LLMs, this section analyzes the compiled text and truncates it to a maximum of 5,000 characters if necessary, optimizing it to prevent token overflow errors.

In [23]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [24]:
def stream_brochure(url):
    company_name = url
    stream = ollama.chat.completions.create(
        model=ollama_model,
        messages=[
            {"role": "user", "content": brochure_system_prompt + get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

### User Interface with Gradio
In this section, a simple and user-friendly interface is designed using the Gradio framework. The user can easily enter their company's website URL and view/download the AI-generated brochure output in text or file format.

In [ ]:
message_input = gr.Textbox(label="Your message:", info="Enter a URL for GPT to analyze", lines=1)
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_brochure,
    title="GPT", 
    inputs=[message_input], 
    outputs=[message_output], 
    examples=[
        "https://aroontech.com",
        "https://openai.com",
        ], 
    flagging_mode="never"
    )
view.launch()

* Running on local URL:  http://127.0.0.1:7878
* To create a public link, set `share=True` in `launch()`.


Selecting relevant links for https://aroontech.com by calling llama3.2
Generated user prompt for relevant links for https://aroontech.com

Found 4 relevant links

Finished fetching contents for home/directions at https://aroontech.com/home/directions
Finished fetching contents for investors page at https://aroontech.com/investors
Finished fetching contents for company at https://aroontech.com/company/the-technology
Finished fetching contents for careers page at https://aroontech.com/careers
Finished fetching contents for all relevant links https://aroontech.com and sublinks
